# AutoEIT Audio Transcription — GSoC 2026 Test I

**Author:** GSoC 2026 Applicant  
**Task:** Transcribe 4 Spanish EIT (Elicited Imitation Task) audio files using OpenAI Whisper, producing a completed Excel file matching the provided template.

---

## 1. Background & Task Overview

The **Elicited Imitation Task (EIT)** is a validated measure of second-language proficiency (Ortega 2000; Faretta-Stutenberg et al., 2023). Participants listen to 30 stimulus sentences and immediately try to repeat them verbatim. Transcriptions must capture the participant's *exact production*, including:

- Disfluencies (uh, eh, false starts, repetitions)
- Grammar/vocabulary errors (NOT corrected — they reflect proficiency)
- Only **ASR errors** (e.g., wrong Spanish word due to acoustic confusion) are corrected

### Files Processed
| Sheet | Audio File | Skip (intro) | Notes |
|-------|-----------|-------------|-------|
| 38010-2A | 038010_EIT-2A.mp3 | 150 s | EIT Version 2A |
| 38011-1A | 038011_EIT-1A.mp3 | 150 s | EIT Version 1A |
| 38012-2A | 038012_EIT-2A.mp3 | 720 s | 12-min skip (documented issue) |
| 38015-1A | 038015_EIT-1A.mp3 | 150 s | EIT Version 1A |


## 2. Approach & Methodology

### 2.1 ASR Engine: OpenAI Whisper

We chose **Whisper `medium`** (769M parameters) for the following reasons:

- **Language support**: Whisper has strong Spanish ASR performance, trained on 680,000 hours of multilingual audio.
- **CPU-friendly**: Medium balances accuracy vs. speed without requiring a GPU.
- **Disfluency handling**: Unlike many commercial ASR systems, Whisper does not auto-correct speech; it transcribes what it hears.
- **Segment timestamps**: Word-level timestamps allow precise segmentation.

### 2.2 Transcription Parameters

```python
model.transcribe(
    audio,
    language="es",                    # Force Spanish (avoid language detection errors)
    task="transcribe",                # Transcription, not translation
    initial_prompt=context,           # Primes model with EIT vocabulary
    word_timestamps=True,             # Enables precise segmentation
    condition_on_previous_text=True,  # Maintains coherence across segments
    no_speech_threshold=0.6,          # Filters silence/pauses
    compression_ratio_threshold=2.4,  # Filters hallucinated repetitions
    temperature=(0.0, 0.2, 0.4),      # Greedy decoding, fallback to sampling
)
```

**Context prompt strategy**: We prepend a Spanish EIT description + first 5 target sentences. This:
- Anchors Whisper's language model to Spanish EIT vocabulary
- Reduces hallucination of English or music tokens
- Does NOT constrain output (the participant may differ from target)

### 2.3 Audio Preprocessing

- **Skip intro**: Each file begins with ~2:30 of instructions and practice items. We skip these seconds of audio using numpy array slicing (avoids re-encoding, no quality loss).
- **Participant 038012 exception**: 12:00 skip due to a documented recording issue.
- **Sample rate**: Whisper natively uses 16 kHz mono; `whisper.load_audio()` handles conversion via ffmpeg.

### 2.4 Sentence Segmentation

EIT audio has a repeating structure: `[stimulus plays] → [pause] → [participant repeats]`. Whisper segments on natural pauses. Our algorithm:

1. Collect all Whisper segments with timing
2. Group segments with inter-segment gap < 3.0 s → single utterance
3. If groups ≈ 2× expected (60), label alternating as stimulus/response using fuzzy similarity to target sentences
4. If groups < 30, split unusually long segments
5. Ensure exactly 30 sentences; pad missing with `[inaudible]`

### 2.5 Transcription Cleaning

Rules applied (`clean_transcription()`):
- Remove Whisper hallucination patterns (repeated phrase at end)
- Remove `[music]` / `(música)` artifacts
- Normalize whitespace
- **Preserve**: disfluencies, grammar errors, vocabulary errors, false starts


## 3. Running the Transcription

In [ ]:
import os, json, openpyxl

BASE_DIR = r"c:\Users\neeri\Desktop\Gsoc_2"
OUTPUT_FILE = os.path.join(BASE_DIR, "AutoEIT Sample Audio for Transcribing_COMPLETED.xlsx")
RAW_DIR = os.path.join(BASE_DIR, "raw_transcriptions")

print("Output Excel exists:", os.path.exists(OUTPUT_FILE))
print("Raw transcriptions dir exists:", os.path.exists(RAW_DIR))
if os.path.exists(RAW_DIR):
    print("Raw files:", os.listdir(RAW_DIR))

## 4. Output Verification

In [ ]:
# Load completed Excel and display transcription counts
wb = openpyxl.load_workbook(OUTPUT_FILE)
PLACEHOLDER_LABELS = {"[inaudible]", "[no speech detected]", "[file not found]"}

for sheet in wb.sheetnames:
    ws = wb[sheet]
    sentences = [ws.cell(row=r, column=3).value for r in range(2, 32)]
    filled = sum(1 for s in sentences if s and s not in PLACEHOLDER_LABELS)
    print(f"Sheet {sheet}: {filled}/30 sentences transcribed")

In [ ]:
# Display all transcriptions for a participant — change sheet_name to inspect others
sheet_name = "38010-2A"
ws = wb[sheet_name]

TARGET_SENTENCES = [
    "Quiero cortarme el pelo", "El libro está en la mesa", "El carro lo tiene Pedro",
    "El se ducha cada mañana", "¿Qué dice usted que va a hacer hoy?",
    "Dudo que sepa manejar muy bien", "Las calles de esta ciudad son muy anchas",
    "Puede que llueva mañana todo el día", "Las casas son muy bonitas pero caras",
    "Me gustan las películas que acaban bien", "El chico con el que yo salgo es español",
    "Después de cenar me fui a dormir tranquilo", "Quiero una casa en la que vivan mis animales",
    "A nosotros nos fascinan las fiestas grandiosas", "Ella sólo bebe cerveza y no come nada",
    "Me gustaría que el precio de las casas bajara", "Cruza a la derecha y después sigue todo recto",
    "Ella ha terminado de pintar su apartamento", "Me gustaría que empezara a hacer más calor pronto",
    "El niño al que se le murió el gato está triste", "Una amiga mía cuida a los niños de mi vecino",
    "El gato que era negro fue perseguido por el perro", "Antes de poder salir él tiene que limpiar su cuarto",
    "La cantidad de personas que fuman ha disminuido", "Después de llegar a casa del trabajo tomé la cena",
    "El ladrón al que atrapó la policía era famoso", "Le pedí a un amigo que me ayudara con la tarea",
    "El examen no fue tan difícil como me habían dicho",
    "¿Serías tan amable de darme el libro que está en la mesa?",
    "Hay mucha gente que no toma nada para el desayuno",
]

print(f"{'#':<4} {'TARGET':<50} {'TRANSCRIPTION'}")
print("-"*100)
for i in range(30):
    target = TARGET_SENTENCES[i] if i < len(TARGET_SENTENCES) else ""
    transcription = ws.cell(row=i+2, column=3).value or "[empty]"
    print(f"{i+1:<4} {target:<50} {transcription}")

## 5. Spot-check: Comparing to Provided Example Transcription

In [ ]:
from difflib import SequenceMatcher

# Load an existing raw JSON to analyze segment distribution
raw_files = [f for f in os.listdir(RAW_DIR) if f.endswith('.json')]

for rf in raw_files:
    with open(os.path.join(RAW_DIR, rf), encoding='utf-8') as f:
        data = json.load(f)
    segs = data['segments']
    print(f"\n{rf}")
    print(f"  Total segments: {len(segs)}")
    print(f"  Total text length: {len(data['text'])} chars")
    if segs:
        gaps = [segs[i]['start'] - segs[i-1]['end'] for i in range(1, len(segs))]
        print(f"  Avg inter-segment gap: {sum(gaps)/len(gaps):.2f} s")
        print(f"  Max gap: {max(gaps):.2f} s")

## 6. Evaluation & Challenges

### 6.1 ASR vs. Human Transcription Trade-offs

| Aspect | Whisper Behaviour | EIT Requirement |
|--------|-----------------|------------------|
| Disfluencies | Usually preserved | **Must preserve** |
| Filler words (eh, uh) | Sometimes dropped | Should keep |
| Near-native production | High accuracy | Low error rate |
| Learner errors | May 'correct' to target | **Must NOT correct** |
| Background noise | Occasional hallucination | Filter with `no_speech_threshold` |

### 6.2 Segmentation Strategy

The main challenge was isolating participant responses from stimulus playback. The 3-second gap threshold was chosen empirically based on typical EIT timing (stimulus ~5–15 words, participant response immediately follows, then ~5–8 second gap before next item).

### 6.3 Known Limitations

1. **CPU-only inference**: Whisper medium on CPU runs at ~3–5× real-time. A GPU would reduce this to <1× real-time.
2. **Disfluency dropout**: Whisper occasionally omits very short fillers (<200ms). Human review of flagged items would catch these.
3. **Stimulus bleed**: If participant begins repeating while stimulus is still fading, both may be captured in one segment.
4. **Participant 038012 12:00 skip**: The special offset means we may miss the first few experimental items if they fall before the skip point. This is documented per the task instructions.

### 6.4 Quality Assurance Approach

- Raw Whisper output saved as JSON for audit trail
- `[inaudible]` placeholder used rather than guessing when segment confidence is low
- Human reviewer should cross-check any sentence where transcription differs substantially from the target by >50% edit distance


In [ ]:
# Flag low-confidence transcriptions (very different from target)
def similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

print("Sentences flagged for human review (similarity < 0.3 to target):")
print(f"{'Sheet':<12} {'#':<4} {'Similarity':<12} {'Transcription'}")
print("-"*80)

for sheet in wb.sheetnames:
    ws = wb[sheet]
    for i in range(30):
        t = ws.cell(row=i+2, column=3).value or ""
        if not t or t in PLACEHOLDER_LABELS:
            print(f"{sheet:<12} {i+1:<4} {'N/A':<12} {t or '[empty]'}")
            continue
        sim = similarity(t, TARGET_SENTENCES[i])
        if sim < 0.3:
            print(f"{sheet:<12} {i+1:<4} {sim:<12.2f} {t[:60]}")

## 7. Summary

- **Model**: OpenAI Whisper `medium`, Spanish, word timestamps
- **4 participants** transcribed, 30 sentences each
- **Output**: `AutoEIT Sample Audio for Transcribing_COMPLETED.xlsx` (columns: Item # | Target | Transcription)
- **Raw data**: `raw_transcriptions/` directory (JSON per participant)
- **Key design decisions**: preserve disfluencies, use target sentences as context prompt, gap-based segmentation, similarity-based stimulus filtering

This transcription serves as input to the AutoEIT scoring pipeline (Test II), which will automatically score each response using meaning-based rubrics.